# qang — start here: 15 quantum gates and 14 algorithms in qg units

This is the **starter notebook** for the `qang` library (`pip install qang`). It runs, in code, the qg formulation of the standard gates and algorithms (article: *Quantum gates and algorithms in qg units: Hamming-weight conservation and readout classes*, `manuscript/qang_formulation.pdf`).

**The idea in one line.** An $n$-qubit state is described by its **qg values**, $qg_P=\langle P\rangle$ for every Pauli string $P$ (for one qubit: $qg_X, qg_Y, qg_Z$, with $qg_Z=\cos\theta$), and a gate $U$ acts as

$$qg'_P = qg_{U^\dagger P U}.$$

With that rule every gate becomes a table of qg transformations, and every algorithm a **readout identity**: which qg values hold the answer.

**What it is and what it is not.** This is the Pauli (Heisenberg) representation, which is known, written in the qg units used throughout the project. It adds a common language and two practical classifications: which gates conserve Hamming weight (where the qg symmetry witness and filter apply), and which algorithms are read out from a few qg values and which need the full histogram. It is not a new quantum advantage, and everything here is exact simulation.

**Contents**
0. Installation
1. The framework: qg values of a state
2. The 15 gates
3. The 14 algorithms
4. Summary: what is read with what, and where to go next

*A Spanish version of this notebook is `notebooks/qang_inicio_formulacion.ipynb`.*

## 0. Installation

In [1]:
# In Colab: installs qang from PyPI (0.6.4 or later ships the radius functions).
import importlib, subprocess, sys
def _ok():
    try:
        import qang.formulation as _F
        return hasattr(_F, "radius_profile")
    except ImportError:
        return False
if not _ok():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang>=0.6.4"], check=False)
    importlib.invalidate_caches()
    if not _ok():  # fallback: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/Viny2030/qang.git"], check=True)
        importlib.invalidate_caches()
import numpy as np
import qang
from qang import formulation as F
print("qang", qang.__version__)

qang 0.6.4


## 1. The framework: qg values of a state

`F.qg_values(state)` returns every non-zero $qg_P$; `F.state_from_qg` goes back, $\rho=2^{-n}\sum_P qg_P\,P$. The Born rule is $P(0)=(1+qg_Z)/2$.

In [2]:
ket0 = np.array([1, 0])
print("|0>        :", F.qg_values(ket0))
theta = 0.7
print("Ry(0.7)|0> :", F.qg_values(F.ry(theta) @ ket0), "| cos(0.7) =", round(np.cos(theta), 12), "-> the qang identity qg_Z = cos(theta)")
print("P(0) from qg_Z:", round(F.born_p0(np.cos(theta)), 4))
bell = F.apply_gate(F.apply_gate(F.qg_values(np.eye(4)[0]), np.kron(F.H, F.I2)), F.CX)
print("Bell (H then CNOT on |00>):", bell)
print("back to the density matrix:\n", np.round(F.state_from_qg(bell, 2).real, 3))

|0>        : {'I': 1.0, 'Z': 1.0}
Ry(0.7)|0> : {'I': 1.0, 'X': 0.644217687238, 'Z': 0.764842187284} | cos(0.7) = 0.764842187284 -> the qang identity qg_Z = cos(theta)
P(0) from qg_Z: 0.8824
Bell (H then CNOT on |00>): {'II': 1.0, 'XX': 1.0, 'YY': -1.0, 'ZZ': 1.0}
back to the density matrix:
 [[0.5 0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0.5]]


**Reading.** A Bell state has no local qg value at all (every $qg_Z$, $qg_X$ is 0): all its information is in the correlations $qg_{XX}=1$, $qg_{YY}=-1$, $qg_{ZZ}=1$.

## 2. The 15 gates

`F.gate_table(U)` gives, for each Pauli of each qubit, the combination $U^\dagger P U$: the new qg value on the left is that combination of old qg values. Convention: the leftmost letter of a string is qubit 0 (the control of CX and CZ).

In [3]:
def show(name, U):
    tab = F.gate_table(U)
    pretty = {p: " ".join(f"{c:+.4g}·qg_{q}" for q, c in v.items()) for p, v in tab.items() if v != {p: 1.0}}
    n = int(np.log2(U.shape[0]))
    tag = "Clifford" if F.is_clifford(U) else "non-Clifford"
    print(f"{name:11s} [{tag}, conserves weight: {F.conserves_weight(U, n)}]")
    for p, s in pretty.items():
        print(f"     qg'_{p} = {s}")

t = 0.7
for name, U in [("1. X", F.X), ("2. Y", F.Y), ("3. Z", F.Z), ("4. H", F.H), ("5. S", F.S), ("6. T", F.T),
                ("7. Rx(0.7)", F.rx(t)), ("8. Ry(0.7)", F.ry(t)), ("9. Rz(0.7)", F.rz(t))]:
    show(name, U)

1. X        [Clifford, conserves weight: False]
     qg'_Y = -1·qg_Y
     qg'_Z = -1·qg_Z
2. Y        [Clifford, conserves weight: False]
     qg'_X = -1·qg_X
     qg'_Z = -1·qg_Z
3. Z        [Clifford, conserves weight: True]
     qg'_X = -1·qg_X
     qg'_Y = -1·qg_Y
4. H        [Clifford, conserves weight: False]
     qg'_X = +1·qg_Z
     qg'_Y = -1·qg_Y
     qg'_Z = +1·qg_X
5. S        [Clifford, conserves weight: True]
     qg'_X = -1·qg_Y
     qg'_Y = +1·qg_X
6. T        [non-Clifford, conserves weight: True]
     qg'_X = +0.7071·qg_X -0.7071·qg_Y
     qg'_Y = +0.7071·qg_X +0.7071·qg_Y
7. Rx(0.7)  [non-Clifford, conserves weight: False]
     qg'_Y = +0.7648·qg_Y -0.6442·qg_Z
     qg'_Z = +0.6442·qg_Y +0.7648·qg_Z
8. Ry(0.7)  [non-Clifford, conserves weight: False]
     qg'_X = +0.7648·qg_X +0.6442·qg_Z
     qg'_Z = -0.6442·qg_X +0.7648·qg_Z
9. Rz(0.7)  [non-Clifford, conserves weight: True]
     qg'_X = +0.7648·qg_X -0.6442·qg_Y
     qg'_Y = +0.6442·qg_X +0.7648·qg_Y


In [4]:
for name, U in [("10. CNOT", F.CX), ("11. CZ", F.CZ), ("12. SWAP", F.SWAP), ("13. iSWAP", F.ISWAP),
                ("14. Toffoli", F.TOFFOLI), ("15. Fredkin", F.FREDKIN)]:
    show(name, U)

10. CNOT    [Clifford, conserves weight: False]
     qg'_XI = +1·qg_XX
     qg'_YI = +1·qg_YX
     qg'_IY = +1·qg_ZY
     qg'_IZ = +1·qg_ZZ
11. CZ      [Clifford, conserves weight: True]
     qg'_XI = +1·qg_XZ
     qg'_YI = +1·qg_YZ
     qg'_IX = +1·qg_ZX
     qg'_IY = +1·qg_ZY
12. SWAP    [Clifford, conserves weight: True]
     qg'_XI = +1·qg_IX
     qg'_YI = +1·qg_IY
     qg'_ZI = +1·qg_IZ
     qg'_IX = +1·qg_XI
     qg'_IY = +1·qg_YI
     qg'_IZ = +1·qg_ZI
13. iSWAP   [Clifford, conserves weight: True]
     qg'_XI = -1·qg_ZY
     qg'_YI = +1·qg_ZX
     qg'_ZI = +1·qg_IZ
     qg'_IX = -1·qg_YZ
     qg'_IY = +1·qg_XZ
     qg'_IZ = +1·qg_ZI


14. Toffoli [non-Clifford, conserves weight: False]
     qg'_XII = +0.5·qg_XII +0.5·qg_XIX +0.5·qg_XZI -0.5·qg_XZX
     qg'_YII = +0.5·qg_YII +0.5·qg_YIX +0.5·qg_YZI -0.5·qg_YZX
     qg'_IXI = +0.5·qg_IXI +0.5·qg_IXX +0.5·qg_ZXI -0.5·qg_ZXX
     qg'_IYI = +0.5·qg_IYI +0.5·qg_IYX +0.5·qg_ZYI -0.5·qg_ZYX
     qg'_IIY = +0.5·qg_IIY +0.5·qg_IZY +0.5·qg_ZIY -0.5·qg_ZZY
     qg'_IIZ = +0.5·qg_IIZ +0.5·qg_IZZ +0.5·qg_ZIZ -0.5·qg_ZZZ


15. Fredkin [non-Clifford, conserves weight: True]
     qg'_XII = +0.5·qg_XII +0.5·qg_XXX +0.5·qg_XYY +0.5·qg_XZZ
     qg'_YII = +0.5·qg_YII +0.5·qg_YXX +0.5·qg_YYY +0.5·qg_YZZ
     qg'_IXI = +0.5·qg_IIX +0.5·qg_IXI -0.5·qg_ZIX +0.5·qg_ZXI
     qg'_IYI = +0.5·qg_IIY +0.5·qg_IYI -0.5·qg_ZIY +0.5·qg_ZYI
     qg'_IZI = +0.5·qg_IIZ +0.5·qg_IZI -0.5·qg_ZIZ +0.5·qg_ZZI
     qg'_IIX = +0.5·qg_IIX +0.5·qg_IXI +0.5·qg_ZIX -0.5·qg_ZXI
     qg'_IIY = +0.5·qg_IIY +0.5·qg_IYI +0.5·qg_ZIY -0.5·qg_ZYI
     qg'_IIZ = +0.5·qg_IIZ +0.5·qg_IZI +0.5·qg_ZIZ -0.5·qg_ZZI


**Reading.**
* **Clifford or not.** X, Y, Z, H, S, CNOT, CZ, SWAP and iSWAP only permute qg values with signs; T, generic rotations, Toffoli and Fredkin **mix** them (one string opens into 2 or 4). That growth is what makes non-Clifford circuits hard to simulate.
* **Z-basis blindness.** Z, S, T, $R_z$ and CZ leave every $qg_Z$ fixed: their effect shows up only after a Hadamard.
* **CNOT** turns a correlation into a local value: the target's $qg_Z$ becomes the parity $qg_{ZZ}$ (this is how a syndrome is extracted).
* **Weight conservation.** CZ, SWAP, iSWAP and Fredkin conserve Hamming weight; CNOT and Toffoli do not. Circuits built from weight-conserving gates admit the qg witness and filter.

In [5]:
# Toffoli checked on a random 3-qubit state: qg'_IIZ = (qg_IIZ + qg_IZZ + qg_ZIZ - qg_ZZZ)/2
rng = np.random.default_rng(1)
psi = rng.normal(size=8) + 1j * rng.normal(size=8); psi /= np.linalg.norm(psi)
q = F.qg_values(psi); q2 = F.qg_values(F.TOFFOLI @ psi)
g = lambda d, k: d.get(k, 0.0)
print("qg'_IIZ measured :", g(q2, "IIZ"))
print("table formula    :", round((g(q, "IIZ") + g(q, "IZZ") + g(q, "ZIZ") - g(q, "ZZZ")) / 2, 12))

qg'_IIZ measured : -0.183017194579
table formula    : -0.183017194578


In [6]:
# With Qiskit (optional): the native SignedRQangGate is Ry(s·arccos qg_Z) and gives exactly that qg_Z.
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    from qang.qiskit_gate import SignedRQangGate
    for qz in (0.9, 0.0, -0.6):
        qc = QuantumCircuit(1); qc.append(SignedRQangGate(qz, sign=-1), [0])
        v = F.qg_values(Statevector(qc).data)
        print(f"SignedRQangGate({qz:+.1f}, s=-1): qg_Z = {v.get('Z', 0):+.3f}, qg_X = {v.get('X', 0):+.3f}")
except ImportError:
    print("Qiskit is not installed: pip install 'qang[qiskit]'")

SignedRQangGate(+0.9, s=-1): qg_Z = +0.900, qg_X = -0.436
SignedRQangGate(+0.0, s=-1): qg_Z = +0.000, qg_X = -1.000
SignedRQangGate(-0.6, s=-1): qg_Z = -0.600, qg_X = -0.800


## 3. The 14 algorithms

For each: the readout identity in qg, checked on the exact state.

In [7]:
# 1. Deutsch-Jozsa and 2. Bernstein-Vazirani: the local qg_Z are enough
n = 4
const = F.phase_oracle_output(np.ones(2**n))
f_bal = np.array([1, -1] * 8)
print("1. DJ  constant ->", F.deutsch_jozsa_is_constant(const, n), "| balanced ->", F.deutsch_jozsa_is_constant(F.phase_oracle_output(f_bal), n))
s = np.array([1, 0, 1, 1])
xs = (np.arange(2**n)[:, None] >> np.arange(n)[::-1]) & 1
p_bv = F.phase_oracle_output((-1.0) ** (xs @ s))
print("2. BV  qg_Z per qubit:", [round(v, 3) for v in F.single_qg_z(p_bv, n)], "-> secret", F.bernstein_vazirani_secret(p_bv, n))

1. DJ  constant -> True | balanced -> False
2. BV  qg_Z per qubit: [-1.0, 1.0, -1.0, -1.0] -> secret [1, 0, 1, 1]


In [8]:
# 3. Simon: the only non-trivial non-zero Z-parity is the secret
print("3. Simon, s = 101: non-zero parities", F.simon_nonzero_parities(F.simon_output([1, 0, 1]), 3))
# 4. Shor (order finding): for r = 4 the local qg_Z show it; for r = 6 they do not
for r in (4, 6):
    print(f"4. Shor, order {r}: local qg_Z", [round(v, 2) + 0.0 for v in F.single_qg_z(F.order_finding_output(r, 6), 6)])

3. Simon, s = 101: non-zero parities [(0, 0, 0), (1, 0, 1)]
4. Shor, order 4: local qg_Z [0.0, 0.0, 1.0, 1.0, 1.0, 1.0]
4. Shor, order 6: local qg_Z [0.0, 0.33, 0.33, 0.31, 0.25, 0.0]


In [9]:
# 5. Grover: qg_Z of each qubit = (-1)^bit · (N·P - 1)/(N - 1)
n, marked = 5, 19
for k in (1, 2, 4):
    p = F.grover_state(n, marked, k) ** 2
    print(f"5. Grover k={k}: P = {p[marked]:.3f}, qg_Z = {[round(v, 3) for v in F.single_qg_z(p, n)]}, "
          f"formula = {[round(v, 3) for v in F.grover_qg_z_prediction(n, marked, p[marked])]}, marked read = {F.grover_marked_from_signs(p, n)}")

5. Grover k=1: P = 0.258, qg_Z = [-0.234, 0.234, 0.234, -0.234, -0.234], formula = [-0.234, 0.234, 0.234, -0.234, -0.234], marked read = [1, 0, 0, 1, 1]
5. Grover k=2: P = 0.602, qg_Z = [-0.59, 0.59, 0.59, -0.59, -0.59], formula = [-0.59, 0.59, 0.59, -0.59, -0.59], marked read = [1, 0, 0, 1, 1]
5. Grover k=4: P = 0.999, qg_Z = [-0.999, 0.999, 0.999, -0.999, -0.999], formula = [-0.999, 0.999, 0.999, -0.999, -0.999], marked read = [1, 0, 0, 1, 1]


In [10]:
# 6. Phase estimation and 11. phase kickback: the phase lands in the control's equatorial angle, qg_X + i qg_Y = e^{i phi}
phi = 0.9
qx, qy = F.kickback_control(phi)
print(f"11. kickback: qg_X = {qx:.4f}, qg_Y = {qy:.4f}  (cos, sin = {np.cos(phi):.4f}, {np.sin(phi):.4f}) -> phi = {np.arctan2(qy, qx):.4f}")
print("6. iterative QPE: repeating with U^(2^k) reads one bit of the phase at a time, always from the control's qg_X and qg_Y.")

11. kickback: qg_X = 0.6216, qg_Y = 0.7833  (cos, sin = 0.6216, 0.7833) -> phi = 0.9000
6. iterative QPE: repeating with U^(2^k) reads one bit of the phase at a time, always from the control's qg_X and qg_Y.


In [11]:
# 7. HHL: delivers qg values of the solution, and its success probability as the ancilla's polar bias
A = np.array([[2.0, 1.0], [1.0, 3.0]]); b = [1, 0]
qz_anc, ez = F.hhl_readout(A, b, F.Z)
x = np.linalg.solve(A, b); x = x / np.linalg.norm(x)
print(f"7. HHL: ancilla qg_Z = {qz_anc:.4f} (success {(1 - qz_anc) / 2:.3f}); solution qg_Z = {ez:.4f} (direct: {x[0]**2 - x[1]**2:.4f})")

7. HHL: ancilla qg_Z = -0.5279 (success 0.764); solution qg_Z = 0.8000 (direct: 0.8000)


In [12]:
# 8. QFT: a basis state becomes a product of equatorial qubits
n = 3
for x in (3, 5):
    psi = F.qft_matrix(n) @ np.eye(2**n)[x]
    print(f"8. QFT|{x}>:", [tuple(round(v, 3) + 0.0 for v in F.local_qg(psi, n, q)) for q in range(n)], " (qg_X, qg_Y, qg_Z) per qubit")

8. QFT|3>: [(-1.0, 0.0, 0.0), (0.0, -1.0, 0.0), (-0.707, 0.707, 0.0)]  (qg_X, qg_Y, qg_Z) per qubit
8. QFT|5>: [(-1.0, 0.0, 0.0), (0.0, 1.0, 0.0), (-0.707, -0.707, 0.0)]  (qg_X, qg_Y, qg_Z) per qubit


In [13]:
# 9. VQE: the energy is linear in qg values.  10. QAOA MaxCut: the cost comes from ZZ correlations
H_toy = {"ZZ": 1.0, "XX": 0.5, "II": -0.2}
print("9. VQE: energy of the Bell state =", round(F.energy_from_qg(H_toy, bell), 12), "(1 + 0.5 - 0.2)")
tri = [(0, 1, 1.0), (1, 2, 1.0), (0, 2, 1.0)]
for label in ("010", "000"):
    qg3 = F.qg_values(np.eye(8)[int(label, 2)])
    print(f"10. QAOA MaxCut on a triangle, state |{label}>: cut = {F.maxcut_from_qg(tri, qg3, 3)}")

9. VQE: energy of the Bell state = 1.3 (1 + 0.5 - 0.2)
10. QAOA MaxCut on a triangle, state |010>: cut = 2.0
10. QAOA MaxCut on a triangle, state |000>: cut = 0.0


In [14]:
# 12. Quantum counting: one Hadamard test, M = N (1 - qg_X) / 2
qx = F.counting_qg_x(4, [2, 7, 11])
print(f"12. counting: qg_X = {qx:.4f} -> M = {F.count_from_qg_x(qx, 16):.3f} solutions out of 16")
# 13. Single-walker quantum walk: conserves the excitation number, so the register-mean qg_Z stays fixed
try:
    print("13. walk: register-mean qg_Z at t = 0.5 and 2.0:", [round(v, 6) for v in F.walk_register_mean(5, 4, (0.5, 2.0))], "(1 - 2/n = 0.6)")
except ImportError:
    print("13. walk: needs scipy")
# 14. QML: fidelity kernel of product states from the local qg vectors
a = [(np.sin(0.4), 0, np.cos(0.4)), (0, 0, 1)]; b = [(np.sin(1.1), 0, np.cos(1.1)), (1, 0, 0)]
print("14. QML: product kernel =", round(F.product_kernel(a, b), 4), "| Bell purity from qg:", round(F.purity_from_qg(F.state_from_qg(bell, 2), 2), 12))

12. counting: qg_X = 0.6250 -> M = 3.000 solutions out of 16
13. walk: register-mean qg_Z at t = 0.5 and 2.0: [0.6, 0.6] (1 - 2/n = 0.6)
14. QML: product kernel = 0.4412 | Bell purity from qg: 1.0


## 4. Summary: what is read with what

| sufficient readout | algorithms |
|---|---|
| local $qg_Z$ | Deutsch–Jozsa, Bernstein–Vazirani |
| one ancilla's $qg_X$, $qg_Y$ | phase kickback, iterative phase estimation, quantum counting, HHL success |
| few-qubit correlations | QAOA ($qg_{ZZ}$), VQE (Hamiltonian strings) |
| all parities or the histogram | Simon, Shor with general order, Grover (local readout exists but costs $\sim1/P^2$ shots), fidelity kernels |

**Where to go next in the library**
* `qang.core.Qang` and the native gates: `qang.qiskit_gate` (`RQangGate`, `SignedRQangGate`), `qang.cirq_gate`, `qang.pennylane_gate`.
* Few-shot estimation: `qang.statistics.qg_estimate(k0, n_shots, method="bayes")`.
* Sector filter and witness: `qang.sectors.filter_distribution`, `SectorExposurePass`.
* Reproducible research results: `notebooks/qang_avances_colab.ipynb` (Spanish, §20–§93) and `notebooks/qang_full_reference.ipynb`.
* Source and issues: https://github.com/Viny2030/qang

## 5. The local radius: the surface of the sphere (§97)

Each qubit's local qg values are a point in the unit ball. Its squared radius

$$r_q^2 = qg_X^2 + qg_Y^2 + qg_Z^2 = 2\,\mathrm{Tr}\rho_q^2 - 1$$

is 1 on the surface (pure qubit) and 0 at the centre; the qubit lives on a sphere of area $4\pi r_q^2$, and the surface deficit is $4\pi(1-r_q^2)$. If the whole register is pure, $1-r_q^2$ is the known one-tangle of qubit $q$ with the rest (two qubits: the squared concurrence), and its mean is the Meyer–Wallach $Q$. If the register is noisy, the deficit mixes entanglement and noise.

In [15]:
# r^2 = qg_X^2 + qg_Y^2 + qg_Z^2 for each qubit; area 4 pi r^2
ket0 = np.array([1, 0, 0, 0])
bell = np.array([1, 0, 0, 1]) / np.sqrt(2)
mixed = F.state_from_qg({"I": 1, "Z": 0.6}, 1)
for name, s in [("|00>", ket0), ("Bell", bell), ("qubit with qg_Z = 0.6", mixed)]:
    r2 = F.radius_profile(s)
    print(f"{name:22s} r^2 = {r2}   area / 4pi = {[F.sphere_area(x) / (4 * np.pi) for x in r2]}   "
          f"Meyer-Wallach Q = {F.meyer_wallach(s)}")
# for a pure two-qubit state the deficit 1 - r^2 is the squared concurrence
rng = np.random.default_rng(0)
psi = rng.normal(size=4) + 1j * rng.normal(size=4); psi /= np.linalg.norm(psi)
print("random pure state: 1 - r^2 =", F.radius_deficit(psi), " concurrence^2 =",
      round((2 * abs(psi[0] * psi[3] - psi[1] * psi[2])) ** 2, 12))

|00>                   r^2 = [1.0, 1.0]   area / 4pi = [1.0, 1.0]   Meyer-Wallach Q = 0.0
Bell                   r^2 = [0.0, 0.0]   area / 4pi = [0.0, 0.0]   Meyer-Wallach Q = 1.0
qubit with qg_Z = 0.6  r^2 = [0.36]   area / 4pi = [0.36]   Meyer-Wallach Q = 0.64
random pure state: 1 - r^2 = [0.385188575069, 0.385188575069]  concurrence^2 = 0.385188575069


**The 15 gates.** One-qubit gates rotate the sphere and keep every radius; SWAP exchanges radii; CNOT, CZ, iSWAP, Toffoli and Fredkin can take a product input to the centre (deficit 1).

In [16]:
# The 15 gates on the radius (product inputs: stabilizer products + random products)
gates = {**F.GATES_1Q, "Rx(0.7)": F.rx(0.7), "Ry(0.7)": F.ry(0.7), "Rz(0.7)": F.rz(0.7), "P(0.7)": F.phase(0.7),
         **F.GATES_MULTI}
for name, U in gates.items():
    c = F.gate_radius_class(U)
    print(f"{name:8s} {c['class']:10s} max deficit {c['max deficit']}")

X        preserves  max deficit 0.0


Y        preserves  max deficit 0.0


Z        preserves  max deficit 0.0


H        preserves  max deficit 0.0
S        preserves  max deficit 0.0
T        preserves  max deficit 0.0
Rx(0.7)  preserves  max deficit 0.0


Ry(0.7)  preserves  max deficit 0.0


...

CZ       entangling max deficit 1.0


SWAP     permutes   max deficit 0.0


iSWAP    entangling max deficit 1.0


Toffoli  entangling max deficit 1.0


Fredkin  entangling max deficit 1.0


**The 14 algorithms.** Product outputs (Bernstein–Vazirani, QFT of a basis state, constant Deutsch–Jozsa) stay on the surface; Simon and Shor's counting register sit near the centre (the answer is in correlations); Grover entangles and returns to the surface at the optimum; a Hadamard test has $r^2 = |\langle\psi|U|\psi\rangle|^2$, 1 only for an eigenstate.

In [17]:
# Radius of the readout qubits for one instance of each algorithm
for name, r2 in F.algorithm_radii().items():
    if isinstance(r2, dict):
        r2 = {k: round(v[0], 3) for k, v in r2.items()}
    elif isinstance(r2, list):
        r2 = [round(x, 3) for x in r2]
    print(f"{name:26s} {r2}")

Deutsch-Jozsa constant     [1.0, 1.0, 1.0, 1.0]
Deutsch-Jozsa balanced     [0.25, 0.25, 0.25, 0.25]
Bernstein-Vazirani         [1.0, 1.0, 1.0, 1.0]
Simon                      [0.0, 0.0, 0.0]
Shor r=4                   [0.0, 0.0, 1.0, 1.0, 1.0, 1.0]
Shor r=6                   [0.0, 0.11, 0.109, 0.098, 0.088, 0.0]
Grover                     {0: 1.0, 1: 0.821, 2: 0.662, 3: 0.842, 4: 0.998}
kickback eigenstate        1.0
kickback non-eigenstate    0.810804984135
HHL ancilla                0.316718427
QFT                        [1.0, 1.0, 1.0, 1.0]
VQE singlet                [0.0, 0.0]
VQE Ising h=1              [0.8, 0.8]
QAOA ring-4 p=1            [0.25, 0.25, 0.25, 0.25]
counting                   0.390625
walk t=2                   [0.024, 0.871, 0.674, 0.871, 0.024]
kernel product encoding    [1.0, 1.0, 1.0]


**The radius after the qg filter (§98).** In a state of fixed Hamming weight $qg_X = qg_Y = 0$ on every qubit, so $r_q = |qg_Z|$: the deficit is read from Z-basis shots. Under equal T1 the filter keeps the noiseless state, so the filtered deficit is the tangle; without the filter, decay adds a false deficit.

In [18]:
# Weight-conserving circuit under T1, with and without qang (RESEARCH_NOTES §98)
from qang.qml import WeightQNN
from qang.sectors import filter_distribution
from qang.statistics import qg2_unbiased
m = WeightQNN(6, 2, layers=8)                      # 6 qubits, weight 2, depth 24
rng = np.random.default_rng(98)
th = rng.uniform(-np.pi, np.pi, m.n_theta)
psi = np.zeros((1, m.dim)); psi[0, m.idx[2]] = rng.normal(size=len(m.idx[2])); psi /= np.linalg.norm(psi)
p_exact = m.probs(th, psi)[0]
p_noisy = m.probs(th, psi, 0.02)[0]                 # equal T1, gamma = 0.02 per qubit per layer
truth = 1 - (p_exact @ m.zsign) ** 2                 # noiseless tangle of each qubit
with_qang = 1 - (filter_distribution(p_noisy, 6, 2)[0] @ m.zsign) ** 2
without = 1 - (p_noisy @ m.zsign) ** 2
print("noiseless tangle :", np.round(truth, 3))
print("with qang        :", np.round(with_qang, 3), " max error", f"{np.max(np.abs(with_qang - truth)):.1e}")
print("without qang     :", np.round(without, 3), " max error", f"{np.max(np.abs(without - truth)):.3f}")
# from 1000 shots, unbiased estimator (N q^2 - 1)/(N - 1) on the kept shots
c = rng.multinomial(1000, p_noisy / p_noisy.sum())
kept = np.where(m.wt == 2, c, 0)
est = [1 - qg2_unbiased(int(kept @ (m.zsign[:, q] > 0)), int(kept.sum())) for q in range(6)]
print("from shots, qang :", np.round(est, 3), f"({kept.sum()} of 1000 shots kept)")

noiseless tangle : [0.957 0.386 0.873 0.98  0.919 0.742]
with qang        : [0.957 0.386 0.873 0.98  0.919 0.742]  max error 5.6e-16
without qang     : [0.757 0.31  0.658 0.859 0.68  0.527]  max error 0.239
from shots, qang : [0.978 0.399 0.836 0.98  0.902 0.759] (375 of 1000 shots kept)


**Result of the pre-registered study (§98).** Under equal T1 the filtered deficit equals the noiseless tangle to $3\times10^{-15}$; on product states it is exactly 0, while without qang decay shows a false deficit of 0.60–0.95; from shots the filter has the lower error in 30 of 30 configurations (24 of 30 with dephasing, which it does not correct). One prediction failed (the half-filling case). Article: `manuscript/qang_formulation.pdf`, section 6.